# We first need all the datasets related to the province in which we are working like in Islamabad for Now

In [39]:
import osmnx as ox
import folium
import pandas as pd


In [40]:

# # Define Islamabad
# place_name = "Islamabad, Pakistan"

# # Get Islamabad boundary
# gdf = ox.geocode_to_gdf(place_name)
# center = gdf.geometry.centroid.iloc[0]
# lat, lon = center.y, center.x

# # Create folium map
# m = folium.Map(location=[lat, lon], zoom_start=13, tiles='OpenStreetMap')

# # -------------------------------
# # 1. Points of Interest (blue)
# # -------------------------------
# # Get all POIs like shops, offices, schools, etc.
# tags_poi = {"amenity": True, "shop": True, "tourism": True}
# pois = ox.features_from_place(place_name, tags_poi)

# for _, row in pois.iterrows():
#     if row.geometry.geom_type == 'Point':
#         folium.CircleMarker(
#             location=[row.geometry.y, row.geometry.x],
#             radius=3,
#             color='blue',
#             fill=True,
#             fill_opacity=0.6,
#             popup=row.get('name', 'POI')
#         ).add_to(m)

# # -------------------------------
# # 2. Road Network (green lines)
# # -------------------------------
# G = ox.graph_from_place(place_name, network_type='drive')
# edges = ox.graph_to_gdfs(G, nodes=False, edges=True)

# for _, row in edges.iterrows():
#     coords = list(row.geometry.coords)
#     folium.PolyLine(locations=[(lat, lon) for lon, lat in coords], color='green', weight=1.5).add_to(m)

# # -------------------------------
# # 3. Non-constructive areas (red)
# # -------------------------------
# # These include parks, lakes, etc.
# tags_nca = {
#     "leisure": ["park", "nature_reserve", "garden"],
#     "landuse": ["forest", "recreation_ground", "grass", "meadow"],
#     "natural": ["water", "wetland"]
# }
# nca = ox.features_from_place(place_name, tags_nca)

# for _, row in nca.iterrows():
#     if row.geometry.geom_type in ['Polygon', 'MultiPolygon']:
#         folium.GeoJson(
#             row.geometry,
#             style_function=lambda x: {'fillColor': 'red', 'color': 'red', 'weight': 1, 'fillOpacity': 0.5}
#         ).add_to(m)

# # -------------------------------
# # Save or show
# # -------------------------------
# m.save("islamabad_visualized.html")

In [41]:
#m

In [42]:
place = "Islamabad, Pakistan"
tags = {
    'amenity': ['restaurant', 'fast_food', 'school', 'bank', 'hospital', 'cafe'],
    'shop': True
}

# Make sure osmnx version is 1.1.0 or later
pois = ox.features_from_place(place, tags)  # ✅ correct method in latest version
pois[['geometry', 'name', 'amenity', 'shop']].to_csv('pois.csv')

In [43]:
df = pd.read_csv("pois.csv")

In [44]:
df.head(10)

,element,id,geometry,name,amenity,shop
0,node,61146664,POINT (73.0112954 33.698086),عسکری بینک,bank,NaN
1,node,61146673,POINT (73.0105887 33.6976525),الائیڈ بینک لمیٹڈ,bank,NaN
2,node,61146691,POINT (73.0118772 33.6960411),United Bank Ltd,bank,NaN
3,node,211325636,POINT (73.0570403 33.7383666),Daman-e-Koh,restaurant,NaN
4,node,212538228,POINT (72.986826 33.683766),اسٹینڈرڈ چارٹرڈ,bank,NaN
5,node,234511823,POINT (73.0113365 33.6966838),اسٹینڈرڈ چارٹرڈ,bank,NaN
6,node,237459304,POINT (72.9887772 33.684909),City Bank,bank,NaN
7,node,237492791,POINT (72.9868747 33.6838147),Saudi Pak Commercial Bank,bank,NaN
8,node,237495339,POINT (72.9860121 33.6843984),عسکری بینک,bank,NaN
9,node,240218764,POINT (73.0417644 33.7104738),Ali Medical Centre,hospital,NaN


In [45]:
df.sample(2)

,element,id,geometry,name,amenity,shop
15,node,254975704,POINT (73.0576782 33.7259475),The Hot Spot,cafe,NaN
1499,node,11056628891,POINT (73.0648387 33.6990715),Zia Balti & BBQ,restaurant,NaN


In [46]:
import pandas as pd
from shapely import wkt
from shapely.geometry import Point, Polygon

# Load your raw pois.csv
df = pd.read_csv("pois.csv")

# Drop rows where both amenity and shop are missing
df = df.dropna(subset=["amenity", "shop"], how='all')

# Merge amenity and shop into one clean 'category' column
df["category"] = df["amenity"].fillna(df["shop"])

# Drop original columns (optional)
df = df.drop(columns=["amenity", "shop"])

# Convert geometry WKT to shapely objects
df["geometry"] = df["geometry"].apply(wkt.loads)

# Extract lat/lon from geometry
def get_lat(geom):
    if isinstance(geom, Point):
        return geom.y
    elif isinstance(geom, Polygon):
        return geom.centroid.y
    return None

def get_lon(geom):
    if isinstance(geom, Point):
        return geom.x
    elif isinstance(geom, Polygon):
        return geom.centroid.x
    return None

df["lat"] = df["geometry"].apply(get_lat)
df["lon"] = df["geometry"].apply(get_lon)

# Drop geometry if you don't need full shape anymore
df = df.drop(columns=["geometry"])

# Optional: rename or reorder columns
df = df[["element", "id", "name", "category", "lat", "lon"]]

# Save cleaned POIs
df.to_csv("pois_cleaned.csv", index=False)

print("✅ Cleaned POIs (points + polygons) saved as pois_cleaned.csv")


✅ Cleaned POIs (points + polygons) saved as pois_cleaned.csv


In [47]:
# Define the area — you can change this to "Taxila, Pakistan" or a bounding box
place_name = "Islamabad, Pakistan"

# Extract drivable road network graph
G = ox.graph_from_place(place_name, network_type='drive')

# Convert to GeoDataFrame (only edges — roads)
edges = ox.graph_to_gdfs(G, nodes=False)

# Optional: Select only useful columns to save
columns_to_save = ['length', 'highway', 'geometry']
edges = edges[columns_to_save]

# Save to CSV
edges.to_csv("road_network_edges.csv", index=False)

print("✅ Road network saved to road_network_edges.csv")


✅ Road network saved to road_network_edges.csv


In [48]:
import osmnx.features as oxf  # ✅ required for feature extraction

tags = {'landuse': True}

# Extract land use geometries
landuse = oxf.features_from_place(place_name, tags)

# Optional: Select relevant columns
landuse_filtered = landuse[['landuse', 'geometry']].copy()

# Save to CSV
landuse_filtered.to_csv("landuse.csv", index=False)

print("✅ Land use data saved to landuse.csv")

✅ Land use data saved to landuse.csv


In [49]:
new_tags = {'building': True}

buildings = oxf.features_from_place(place_name, new_tags)

fil = buildings[['building', 'geometry']].copy()
fil.to_csv('buildings_data.csv', index = False)


In [50]:
import shutil

shutil.rmtree("C:/Users/tehma/OneDrive/Desktop/MLP/New_Best_Shop_Location_Prediction_Model/Jupyter_Notebooks_VsCode/SHOP_PROJECT/cache")

In [51]:
df_new = pd.read_csv("pois_cleaned.csv")
df_new.head()

,element,id,name,category,lat,lon
0,node,61146664,عسکری بینک,bank,33.698086,73.011295
1,node,61146673,الائیڈ بینک لمیٹڈ,bank,33.697652,73.010589
2,node,61146691,United Bank Ltd,bank,33.696041,73.011877
3,node,211325636,Daman-e-Koh,restaurant,33.738367,73.057040
4,node,212538228,اسٹینڈرڈ چارٹرڈ,bank,33.683766,72.986826


In [52]:
df_new.sample(10)

,element,id,name,category,lat,lon
1074,node,6195714188,Silver Bistro,restaurant,33.668247,72.996765
1862,way,1096475323,NaN,kiosk,33.712417,73.087828
1937,way,1308666778,Federal Government Primary School No. 5 G-7/3,school,33.710633,73.073205
1436,node,9942580858,Save Mart,supermarket,33.668497,72.999684
221,node,4417497491,college more,school,33.681107,73.018542
69,node,1341931423,Federal Government Girls College No. 9 G-9/2,school,33.689257,73.026435
84,node,1732632465,Savour Foods,restaurant,33.713218,73.063484
148,node,2993615950,Habibi Restaurant,restaurant,33.666282,73.074071
253,node,4561963301,MCB,bank,33.666234,73.076072
1707,way,23759232,G-7/2 ایف جی گرلز ماڈل اسکول,school,33.704475,73.064078


In [53]:
df_new.describe()

,id,lat,lon
count,1.959000e+03,1959.000000,1959.000000
mean,5.211579e+09,33.676362,73.049516
std,3.208174e+09,0.047749,0.060285
min,3.489900e+06,33.481607,72.826087
25%,4.264745e+09,33.658247,73.011074
50%,4.958570e+09,33.685159,73.049234
75%,6.203933e+09,33.711194,73.076753
max,1.284455e+10,33.784330,73.273174
